In [ ]:
import os, io, zipfile, requests
import numpy as np
import pandas as pd

HORIZONS = [100, 250, 500, 1000, 2000]
STEP = 50                       # 50 ms grid, as in the paper
FOLDER = "TFI data"

def load_trades(day, symbol="BTCUSDT"):
    """Download one day of USDT-M futures aggTrades from data.binance.vision (cached locally)."""
    raw = os.path.join(FOLDER, "raw")
    os.makedirs(raw, exist_ok=True)
    name = f"{symbol}-aggTrades-{day}.zip"
    path = os.path.join(raw, name)
    if not os.path.exists(path):
        url = f"https://data.binance.vision/data/futures/um/daily/aggTrades/{symbol}/{name}"
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)

    with zipfile.ZipFile(path) as z:
        df = pd.read_csv(z.open(z.namelist()[0]), header=None, usecols=[1, 2, 5, 6],
                         names=["p", "q", "T", "m"], dtype=str)
    df = df[df["T"].str.isdigit()]                       # drops the header row if the file has one
    df = df.astype({"p": float, "q": float, "T": np.int64})
    df["m"] = df["m"].str.lower() == "true"
    if df["T"].iloc[0] > 1e14:                           # safety: microseconds → milliseconds
        df["T"] //= 1000
    return df.sort_values("T", kind="stable").reset_index(drop=True)

def compute_features(trades):
    """TFI, PRET and future returns on a 50 ms grid, all computed from trades."""
    T = trades["T"].to_numpy()
    p, q, m = trades["p"].to_numpy(), trades["q"].to_numpy(), trades["m"].to_numpy()
    usd = p * q

    # cumulative sums let us sum any time window with two lookups
    cum = lambda x: np.concatenate([[0.0], np.cumsum(x)])
    C_buy, C_sell = cum(np.where(m, 0.0, usd)), cum(np.where(m, usd, 0.0))
    C_pq, C_q = cum(usd), cum(q)

    grid = np.arange((T[0] // STEP + 1) * STEP, T[-1] + 1, STEP)
    r = np.searchsorted(T, grid, side="right")           # trades with T <= t

    # price: VWAP over [t − 50 ms, t], carried forward when nothing traded (eq. 11)
    l = np.searchsorted(T, grid - STEP, side="left")
    qty = C_q[r] - C_q[l]
    vwap = np.where(qty > 0, (C_pq[r] - C_pq[l]) / np.where(qty > 0, qty, 1), np.nan)
    price = pd.Series(vwap).ffill().to_numpy()

    out = {"t": grid, "price": price}
    for d in HORIZONS:
        l = np.searchsorted(T, grid - d, side="left")    # trades with T >= t − δ
        out[f"TFI_{d}"] = (C_buy[r] - C_buy[l]) - (C_sell[r] - C_sell[l])   # eq. 7, USD
        k = d // STEP
        past = np.full_like(price, np.nan); past[k:] = price[:-k]
        out[f"PRET_{d}"] = (price / past - 1) * 1e4                       # eq. 10, bps
    for h in (500, 1000):
        k = h // STEP
        fut = np.full_like(price, np.nan); fut[:-k] = price[k:]
        out[f"FRET{h}_bps"] = (fut / price - 1) * 1e4                     # target, bps

    df = pd.DataFrame(out)
    keep = (df["t"] >= grid[0] + max(HORIZONS)) & (df["t"] <= grid[-1] - 1000)   # full windows only
    return df[keep].reset_index(drop=True)

def build_tfi_history(start, end, symbol="BTCUSDT"):
    """Download each day in [start, end], compute features, save one file per day."""
    for day in pd.date_range(start, end).strftime("%Y-%m-%d"):
        out = os.path.join(FOLDER, f"features_{symbol}_{day}.parquet")
        if os.path.exists(out):
            print(day, "already done"); continue
        trades = load_trades(day, symbol)
        feats = compute_features(trades)
        feats.to_parquet(out, index=False)
        print(day, f"{len(trades):,} trades → {len(feats):,} rows")

In [ ]:
build_tfi_history("2026-09-20", "2026-09-27")

In [ ]:
from glob import glob
data = pd.concat(pd.read_parquet(f) for f in sorted(glob("TFI data/features_BTCUSDT_*.parquet")))
data["time"] = pd.to_datetime(data["t"], unit="ms", utc=True)

In [ ]:
data

In [ ]:
import numpy as np, pandas as pd
from glob import glob

HORIZONS = [100, 250, 500, 1000, 2000]
files = sorted(glob("TFI data/features_BTCUSDT_*.parquet"))
data = pd.concat(pd.read_parquet(f) for f in files).dropna()
data = data.astype({c: "float32" for c in data.columns if c != "t"})   # halves the memory
data["day"] = pd.to_datetime(data["t"], unit="ms", utc=True).dt.date

days = sorted(data["day"].unique())
calib = data[data["day"] == days[0]]            # day 1: calibrate the TFI transform
train = data[data["day"].isin(days[1:-2])]      # middle days: fit the model
test  = data[data["day"].isin(days[-2:])]       # last 2 days: out-of-sample check
print(f"calibration {days[0]} | train {days[1]}..{days[-3]} | test {days[-2]}..{days[-1]}")

In [ ]:
def bucket_plot(x, y, n=100, q=0.001):
    x, y = np.asarray(x, float), np.asarray(y, float)
    lo, hi = np.quantile(x, [q, 1 - q])
    b = pd.cut(x, np.linspace(lo, hi, n + 1))
    out = pd.DataFrame({"x": x, "y": y}).groupby(b, observed=True).agg(
        tfi=("x", "mean"), fret=("y", "mean"), n=("y", "size"))
    return out[out["n"] >= 100]

bucket_plot(train["TFI_500"], train["FRET500_bps"]).plot.scatter("tfi", "fret", s=8)

In [ ]:
r2 = lambda x, y: np.corrcoef(np.asarray(x, float), np.asarray(y, float))[0, 1] ** 2
y = train["FRET500_bps"]
scores = pd.DataFrame({
    "TFI":  [r2(train[f"TFI_{d}"], y)  for d in HORIZONS],
    "PRET": [r2(train[f"PRET_{d}"], y) for d in HORIZONS]}, index=HORIZONS)
print(scores.round(5))
best_tfi, best_pret = scores["TFI"].idxmax(), scores["PRET"].idxmax()

In [ ]:
def fit_transform(x, y, n=100, q=1e-4):
    """Greedy threshold pruning from the paper: f(x) = sign(x) · #{thresholds ≤ |x|}."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    t = np.linspace(0, np.quantile(np.abs(x), 1 - q), n + 1)[1:]
    k = np.searchsorted(t, np.abs(x), side="right")
    idx = (np.sign(x).astype(int) + 1) * (n + 1) + k
    cnt = np.bincount(idx, minlength=3 * (n + 1)).reshape(3, n + 1)
    sy = np.bincount(idx, weights=y, minlength=3 * (n + 1)).reshape(3, n + 1)
    N, Sy, Syy = len(y), y.sum(), (y * y).sum()
    sgn = np.array([-1, 0, 1])[:, None]

    def score(keep):                      # R² of y on f_keep(x), from bucket totals
        v = sgn * np.concatenate([[0], np.cumsum(keep)])
        Sf, Sff, Sfy = (cnt * v).sum(), (cnt * v * v).sum(), (sy * v).sum()
        vf = Sff - Sf**2 / N
        return (Sfy - Sf * Sy / N) ** 2 / (vf * (Syy - Sy**2 / N)) if vf > 0 else 0.0

    keep = np.ones(n, bool)
    best = start = score(keep)
    while keep.any():                     # drop the threshold that helps most, until nothing helps
        s, i = max((score(np.where(np.arange(n) == i, False, keep)), i) for i in np.flatnonzero(keep))
        if s <= best:
            break
        keep[i], best = False, s
    T = t[keep]
    f = lambda x: np.sign(np.asarray(x, float)) * np.searchsorted(T, np.abs(np.asarray(x, float)), side="right")
    return f, start, best

f_tfi, before, after = fit_transform(calib[f"TFI_{best_tfi}"], calib["FRET500_bps"])
print(f"TFI_{best_tfi}: R² on train  raw {r2(train[f'TFI_{best_tfi}'], y):.5f}  →  transformed {r2(f_tfi(train[f'TFI_{best_tfi}']), y):.5f}")
bucket_plot(f_tfi(train[f"TFI_{best_tfi}"]), y).plot.scatter("tfi", "fret", s=8)

In [ ]:
def features(df):
    return np.column_stack([f_tfi(df[f"TFI_{best_tfi}"]), df[f"PRET_{best_pret}"]]).astype(float)

X_tr = features(train)
mu, sd = X_tr.mean(0), X_tr.std(0)                           # standardise, as the paper does
Z = lambda X: np.column_stack([np.ones(len(X)), (X - mu) / sd])
coef, *_ = np.linalg.lstsq(Z(X_tr), train["FRET500_bps"].to_numpy(float), rcond=None)

pred_tr, pred_te = Z(X_tr) @ coef, Z(features(test)) @ coef
R2 = lambda p, y: 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()
print("coefficients (const, TFI, PRET):", coef.round(4))
print(f"R² in-sample {R2(pred_tr, train['FRET500_bps'].to_numpy(float)):.4f} | "
      f"out-of-sample {R2(pred_te, test['FRET500_bps'].to_numpy(float)):.4f}")

In [ ]:
thr = np.quantile(np.abs(pred_tr), 0.95)          # act only on the strongest 5% of predictions (eq. 23)
yt = test["FRET500_bps"].to_numpy(float)
signal = np.where(pred_te > thr, 1, np.where(pred_te < -thr, -1, 0))
fired = signal != 0
print(f"signal fires {fired.mean():.1%} of the time")
print(f"average move captured: {(signal[fired] * yt[fired]).mean():.2f} bps")
print(f"direction correct: {(np.sign(yt[fired]) == signal[fired]).mean():.1%}")

In [ ]:
data = data.sort_values("t")
g = data.groupby("day")["price"]
data["FRET500_skip"] = (g.shift(-11).astype(float) / g.shift(-1).astype(float) - 1) * 1e4   # t+50 → t+550
tr = data[data["day"].isin(days[1:-2])].dropna(subset=["FRET500_skip"])

for target in ["FRET500_bps", "FRET500_skip"]:
    print(f"{target:13}  TFI (transformed) {r2(f_tfi(tr[f'TFI_{best_tfi}']), tr[target]):.5f}   "
          f"PRET {r2(tr[f'PRET_{best_pret}'], tr[target]):.5f}")

In [ ]:
prices = data["price"].astype(float)
g = prices.groupby(data["day"])
mask = data["day"].isin(days[1:-2])
x_tfi = f_tfi(data.loc[mask, f"TFI_{best_tfi}"])
x_pret = data.loc[mask, f"PRET_{best_pret}"]

for lag in [0, 1, 2, 4, 6, 10, 20]:                        # delay in 50 ms steps
    y = ((g.shift(-(lag + 10)) / g.shift(-lag) - 1) * 1e4)[mask]
    ok = y.notna().to_numpy()
    print(f"delay {lag * 50:>4} ms   TFI {r2(x_tfi[ok], y[ok]):.5f}   PRET {r2(x_pret[ok], y[ok]):.5f}")